In [ ]:
1

In [1]:
!pip install pandas numpy matplotlib sdv sdmetrics scikit-learn openpyxl

   ---------------------------------------- 0.0/207.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/207.9 kB ? eta -:--:--
   ----- --------------------------------- 30.7/207.9 kB 435.7 kB/s eta 0:00:01
   ----- --------------------------------- 30.7/207.9 kB 435.7 kB/s eta 0:00:01
   ------- ------------------------------- 41.0/207.9 kB 178.6 kB/s eta 0:00:01
   ------------- ------------------------- 71.7/207.9 kB 280.5 kB/s eta 0:00:01
   ----------------- --------------------- 92.2/207.9 kB 327.7 kB/s eta 0:00:01
   ----------------- --------------------- 92.2/207.9 kB 327.7 kB/s eta 0:00:01
   ---------------------- --------------- 122.9/207.9 kB 312.9 kB/s eta 0:00:01
   ------------------------------- ------ 174.1/207.9 kB 402.6 kB/s eta 0:00:01
   -------------------------------------  204.8/207.9 kB 461.0 kB/s eta 0:00:01
   -------------------------------------- 207.9/207.9 kB 421.4 kB/s eta 0:00:00
   ---------------------------------------- 0.0/206.8 kB 

In [4]:
# ==========================================
# CERT DATA PREPROCESSING
# FOR SYNTHETIC DATA GENERATION
# ==========================================

import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [6]:
# ==========================================
# FOLDER PATHS
# ==========================================

CERT_FOLDER = Path(r"C:\Users\Admin\Downloads\r4.2\r4.2")

LDAP_FOLDER = CERT_FOLDER / "LDAP"

OUTPUT_FOLDER = Path("data")

OUTPUT_FOLDER.mkdir(exist_ok=True)

print("CERT Folder:", CERT_FOLDER)
print("LDAP Folder:", LDAP_FOLDER)
print("Output Folder:", OUTPUT_FOLDER)

CERT Folder: C:\Users\Admin\Downloads\r4.2\r4.2
LDAP Folder: C:\Users\Admin\Downloads\r4.2\r4.2\LDAP
Output Folder: data


In [8]:
# ==========================================
# FIND FILES
# ==========================================

def find_file(folder, name):

    for file in folder.iterdir():

        if file.stem.lower() == name.lower():

            return file

    return None

In [10]:
# ==========================================
# LOCATE DATA FILES
# ==========================================

logon_file = find_file(CERT_FOLDER, "logon")
device_file = find_file(CERT_FOLDER, "device")
file_file = find_file(CERT_FOLDER, "file")
email_file = find_file(CERT_FOLDER, "email")
http_file = find_file(CERT_FOLDER, "http")

print(logon_file)
print(device_file)
print(file_file)
print(email_file)
print(http_file)

C:\Users\Admin\Downloads\r4.2\r4.2\logon.csv
C:\Users\Admin\Downloads\r4.2\r4.2\device.csv
C:\Users\Admin\Downloads\r4.2\r4.2\file.csv
C:\Users\Admin\Downloads\r4.2\r4.2\email.csv
C:\Users\Admin\Downloads\r4.2\r4.2\http.csv


In [12]:
def load_cert_data(path, action_name, max_rows=50000):

    print(f"Reading {path.name}...")

    # Read only the required rows
    df = pd.read_csv(path, nrows=max_rows, low_memory=False)

    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")
    )

    # Timestamp
    if "date" in df.columns:
        timestamp = df["date"]
    elif "timestamp" in df.columns:
        timestamp = df["timestamp"]
    else:
        timestamp = pd.Series([pd.NaT] * len(df))

    # User
    if "user" in df.columns:
        user = df["user"]
    elif "user_id" in df.columns:
        user = df["user_id"]
    else:
        user = pd.Series([None] * len(df))

    # Location
    if "pc" in df.columns:
        location = df["pc"]
    else:
        location = pd.Series(["Unknown"] * len(df))

    clean = pd.DataFrame({
        "timestamp": timestamp,
        "user_id": user,
        "location": location,
        "action": action_name
    })

    print(f"{action_name}: {len(clean)} rows loaded")

    return clean

In [14]:
logon = load_cert_data(logon_file, "logon", 50000)

device = load_cert_data(device_file, "device", 50000)

file_access = load_cert_data(file_file, "file", 50000)

email = load_cert_data(email_file, "email", 50000)

http = load_cert_data(http_file, "http", 50000)

Reading logon.csv...
logon: 50000 rows loaded
Reading device.csv...
device: 50000 rows loaded
Reading file.csv...
file: 50000 rows loaded
Reading email.csv...
email: 50000 rows loaded
Reading http.csv...
http: 50000 rows loaded


In [16]:
# ==========================================
# COMBINE DATA
# ==========================================

cert = pd.concat([

    logon,

    device,

    file_access,

    email,

    http

], ignore_index=True)

print(cert.shape)

cert.head()

(250000, 4)


,timestamp,user_id,location,action
0,01/02/2010 06:49:00,NGF0157,PC-6056,logon
1,01/02/2010 06:50:00,LRR0148,PC-4275,logon
2,01/02/2010 06:53:04,LRR0148,PC-4124,logon
3,01/02/2010 07:00:00,IRM0931,PC-7188,logon
4,01/02/2010 07:00:00,MOH0273,PC-6699,logon


In [18]:
# ==========================================
# CONVERT DATES
# ==========================================

cert["timestamp"] = pd.to_datetime(

    cert["timestamp"],

    errors="coerce"

)

cert = cert.dropna(subset=["timestamp"])

print(cert.shape)

(250000, 4)


In [20]:
# ==========================================
# FEATURE ENGINEERING
# ==========================================

cert["hour"] = cert["timestamp"].dt.hour

cert["day"] = cert["timestamp"].dt.day

cert["month"] = cert["timestamp"].dt.month

cert["weekday"] = cert["timestamp"].dt.day_name()

cert["after_hours"] = (

    (cert["hour"] < 6)

    |

    (cert["hour"] > 18)

).astype(int)

cert.head()

,timestamp,user_id,location,action,hour,day,month,weekday,after_hours
0,2010-01-02 06:49:00,NGF0157,PC-6056,logon,6,2,1,Saturday,0
1,2010-01-02 06:50:00,LRR0148,PC-4275,logon,6,2,1,Saturday,0
2,2010-01-02 06:53:04,LRR0148,PC-4124,logon,6,2,1,Saturday,0
3,2010-01-02 07:00:00,IRM0931,PC-7188,logon,7,2,1,Saturday,0
4,2010-01-02 07:00:00,MOH0273,PC-6699,logon,7,2,1,Saturday,0


In [22]:
# ==========================================
# LOAD LDAP
# ==========================================

ldap_files = sorted(LDAP_FOLDER.glob("*.csv"))

frames = []

for file in ldap_files:

    temp = pd.read_csv(file)

    temp.columns = (

        temp.columns
        .str.lower()
        .str.strip()
        .str.replace(" ", "_")

    )

    frames.append(temp)

ldap = pd.concat(frames)

ldap = ldap.drop_duplicates(

    subset="user_id",

    keep="last"

)

In [24]:
# ==========================================
# MERGE EMPLOYEE INFO
# ==========================================

cert = cert.merge(

    ldap,

    on="user_id",

    how="left"

)

print(cert.shape)

cert.head()

(250000, 17)


,timestamp,user_id,location,action,hour,day,month,weekday,after_hours,employee_name,email,role,business_unit,functional_unit,department,team,supervisor
0,2010-01-02 06:49:00,NGF0157,PC-6056,logon,6,2,1,Saturday,0,Nissim Gil French,Nissim.Gil.French@dtaa.com,FinancialAnalyst,1,4 - Finance,3 - FinancialPlanning,NaN,Libby Rosalyn Richard
1,2010-01-02 06:50:00,LRR0148,PC-4275,logon,6,2,1,Saturday,0,Libby Rosalyn Richard,Libby.Rosalyn.Richard@dtaa.com,Manager,1,4 - Finance,3 - FinancialPlanning,NaN,Thomas Vladimir Stokes
2,2010-01-02 06:53:04,LRR0148,PC-4124,logon,6,2,1,Saturday,0,Libby Rosalyn Richard,Libby.Rosalyn.Richard@dtaa.com,Manager,1,4 - Finance,3 - FinancialPlanning,NaN,Thomas Vladimir Stokes
3,2010-01-02 07:00:00,IRM0931,PC-7188,logon,7,2,1,Saturday,0,Ignatius Reese Morton,Ignatius.Reese.Morton@dtaa.com,ComputerProgrammer,1,2 - ResearchAndEngineering,2 - SoftwareManagement,6 - Software,Caleb Mason Ball
4,2010-01-02 07:00:00,MOH0273,PC-6699,logon,7,2,1,Saturday,0,Macaulay Otto Hopkins,Macaulay.Otto.Hopkins@dtaa.com,HardwareEngineer,1,2 - ResearchAndEngineering,3 - Engineering,4 - SystemsEngineering,Gage Kaden Odonnell


In [26]:
# ==========================================
# REMOVE DUPLICATES
# ==========================================

cert = cert.drop_duplicates()

print(cert.shape)

(248301, 17)


In [28]:
# ==========================================
# CHECK MISSING VALUES
# ==========================================

print(cert.isnull().sum())

timestamp              0
user_id                0
location               0
action                 0
hour                   0
day                    0
month                  0
weekday                0
after_hours            0
employee_name          0
email                  0
role                   0
business_unit          0
functional_unit      670
department          4420
team               36496
supervisor           490
dtype: int64


In [30]:
# Fill missing categorical values

cert["functional_unit"] = cert["functional_unit"].fillna("Unknown")

cert["department"] = cert["department"].fillna("Unknown")

cert["team"] = cert["team"].fillna("Unknown")

cert["supervisor"] = cert["supervisor"].fillna("Unknown")

In [32]:
print(cert.isnull().sum())

timestamp          0
user_id            0
location           0
action             0
hour               0
day                0
month              0
weekday            0
after_hours        0
employee_name      0
email              0
role               0
business_unit      0
functional_unit    0
department         0
team               0
supervisor         0
dtype: int64


In [34]:
# ==========================================
# SAVE PREPROCESSED DATASET
# ==========================================

cert.to_csv(

    OUTPUT_FOLDER / "cert_preprocessed.csv",

    index=False

)

print("Dataset saved successfully!")

print()

print(cert.head())

Dataset saved successfully!

            timestamp  user_id location action  hour  day  month   weekday  after_hours          employee_name                           email                role  business_unit  \
0 2010-01-02 06:49:00  NGF0157  PC-6056  logon     6    2      1  Saturday            0      Nissim Gil French      Nissim.Gil.French@dtaa.com    FinancialAnalyst              1   
1 2010-01-02 06:50:00  LRR0148  PC-4275  logon     6    2      1  Saturday            0  Libby Rosalyn Richard  Libby.Rosalyn.Richard@dtaa.com             Manager              1   
2 2010-01-02 06:53:04  LRR0148  PC-4124  logon     6    2      1  Saturday            0  Libby Rosalyn Richard  Libby.Rosalyn.Richard@dtaa.com             Manager              1   
3 2010-01-02 07:00:00  IRM0931  PC-7188  logon     7    2      1  Saturday            0  Ignatius Reese Morton  Ignatius.Reese.Morton@dtaa.com  ComputerProgrammer              1   
4 2010-01-02 07:00:00  MOH0273  PC-6699  logon     7    2      1  

In [38]:
import os

print(os.getcwd())

C:\Users\Admin
